# Merge Beam Search Parts (part_00 to part_09)
Combines all 10 beam search output files into a single `eng_swh_beam_M10.jsonl`.

In [1]:
import json
import glob
import os
from pathlib import Path

In [2]:
# Paths
DATA_DIR = Path('../data/synthetic')
OUTPUT_FILE = DATA_DIR / 'eng_swh_beam_M10_merged.jsonl'

# Collect part files in sorted order
part_files = sorted(DATA_DIR.glob('beam_M10_part_*.jsonl'))

print(f'Found {len(part_files)} part files:')
for f in part_files:
    print(f'  {f.name}')

Found 10 part files:
  beam_M10_part_00_000000_010000.jsonl
  beam_M10_part_01_010000_020000.jsonl
  beam_M10_part_02_020000_030000.jsonl
  beam_M10_part_03_030000_040000.jsonl
  beam_M10_part_04_040000_050000.jsonl
  beam_M10_part_05_050000_060000.jsonl
  beam_M10_part_06_060000_070000.jsonl
  beam_M10_part_07_070000_080000.jsonl
  beam_M10_part_08_080000_090000.jsonl
  beam_M10_part_09_090000_100000.jsonl


In [3]:
# Merge all parts
records = []

for part_file in part_files:
    part_records = []
    with open(part_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                part_records.append(json.loads(line))
    print(f'{part_file.name}: {len(part_records):,} records')
    records.extend(part_records)

print(f'\nTotal records: {len(records):,}')

beam_M10_part_00_000000_010000.jsonl: 100,000 records
beam_M10_part_01_010000_020000.jsonl: 100,000 records
beam_M10_part_02_020000_030000.jsonl: 100,000 records
beam_M10_part_03_030000_040000.jsonl: 100,000 records
beam_M10_part_04_040000_050000.jsonl: 100,000 records
beam_M10_part_05_050000_060000.jsonl: 100,000 records
beam_M10_part_06_060000_070000.jsonl: 100,000 records
beam_M10_part_07_070000_080000.jsonl: 100,000 records
beam_M10_part_08_080000_090000.jsonl: 100,000 records
beam_M10_part_09_090000_100000.jsonl: 100,000 records

Total records: 1,000,000


In [4]:
# Quick sanity check — show first and last record
print('First record:')
print(json.dumps(records[0], ensure_ascii=False, indent=2))
print('\nLast record:')
print(json.dumps(records[-1], ensure_ascii=False, indent=2))

First record:
{
  "src_id": 0,
  "src": "And Lewis, who began the season with a half century against Leicestershire, insists he is happy to be able to concentrate on his personal performances rather than worrying about the problems of others.",
  "hyp_id": 0,
  "tgt": "Na Lewis, ambaye alianza msimu huu na nusu karne dhidi ya Leicestershire, anasisitiza kuwa anafurahi kuwa na uwezo wa kuzingatia utendaji wake binafsi badala ya kuhangaikia matatizo ya wengine.",
  "method": "beam",
  "M": 10
}

Last record:
{
  "src_id": 99999,
  "src": "If allegations have been made against you, which are often the case when supervision is ordered, you can visit without fear of any new accusations because there is someone present who can verify what happened during your time together. When using a professional service, you can also be assured that the supervisors are neutral and objective.",
  "hyp_id": 9,
  "tgt": "Ikiwa madai yamefanywa dhidi yako, ambayo mara nyingi hufanyika wakati usimamizi unaamr

In [5]:
# Verify src_id continuity — unique source sentences covered
src_ids = sorted(set(r['src_id'] for r in records))
print(f'Unique src_ids : {len(src_ids):,}')
print(f'src_id range   : {src_ids[0]} → {src_ids[-1]}')

# Check hyp_id distribution per src_id
from collections import Counter
hyp_counts = Counter(r['src_id'] for r in records)
unique_counts = Counter(hyp_counts.values())
print(f'\nHypotheses per src_id (count → occurrences):')
for k, v in sorted(unique_counts.items()):
    print(f'  {k} hyps: {v:,} src_ids')

Unique src_ids : 100,000
src_id range   : 0 → 99999

Hypotheses per src_id (count → occurrences):
  10 hyps: 100,000 src_ids


In [6]:
# Write merged output
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

size_mb = OUTPUT_FILE.stat().st_size / 1024 / 1024
print(f'Saved {len(records):,} records to:')
print(f'  {OUTPUT_FILE}')
print(f'  Size: {size_mb:.1f} MB')

Saved 1,000,000 records to:
  ..\data\synthetic\eng_swh_beam_M10_merged.jsonl
  Size: 400.8 MB
